# 06. Entradas do Modelo 3 (Bayesian GPLVM)

Etapa 6 do fluxo: `01_download.ipynb` → `02_serie_modelos.ipynb` → `03_guia_bases_modelos.ipynb` → `04_entradas_modelo1_gp.ipynb` / `05_entradas_modelo2_rgp.ipynb` / `06_entradas_modelo3_gplvm.ipynb` → notebooks dos modelos (`07`–`09`).

Gera e salva a entrada do GPLVM a partir da `serie` salva no `02_serie_modelos.ipynb`. **Não ajusta o modelo.** O GPLVM recebe janelas de dias de cada estação, montadas dentro de cada bloco (nunca atravessam dois blocos); células sem valor visível viram 0 e a máscara as exclui da verossimilhança. Detalhes na seção 4.3 do `03_guia_bases_modelos.ipynb`.

Saída: `data/exports/<NOME>/modelo3_gplvm/janelas.parquet` (uma linha por janela) e `config.json`.

## 1. Carregar os módulos

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "src" / "main.py").is_file() and (folder / "data" / "usgs").is_dir():
            return folder
    raise FileNotFoundError("Não encontrei src/main.py e data/usgs. Abra o notebook na pasta do projeto.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EXPORT_ROOT
from src.model_data import split_summary
from src.model_exports import (gplvm_table, gplvm_arrays, save_inputs,
                               gplvm_sequence_table, gplvm_sequence_arrays)
from src.forecast import ForecastConfig, forecast_origins, gplvm_forecast_inputs
from src.model_inputs import select_split

## 2. Ler a base

In [2]:
NOME = "serie_2010_2025_200est"   # o mesmo NOME do 02_serie_modelos.ipynb

serie = pd.read_parquet(EXPORT_ROOT / f"{NOME}.parquet")
print("Base:", (EXPORT_ROOT / f"{NOME}_config.json").read_text(encoding="utf-8"))
print(f"{len(serie):,} linhas | {serie['sensor_id'].nunique()} estações")

Base: {
  "selecao": {
    "n_sensors": 200,
    "min_completeness": 0.95,
    "max_zero_frac": 0.05,
    "candidates": null,
    "seed": 42
  },
  "splits": {
    "fracs": [
      0.6,
      0.15,
      0.1,
      0.15
    ],
    "hidden_frac": 0.1,
    "hidden_block_len": 1,
    "log_transform": true,
    "seed": 42
  },
  "periodo": [
    "2010-01-01",
    "2025-12-31"
  ],
  "parametro": "00060",
  "estatistica": "00003"
}
1,168,800 linhas | 200 estações


## 3. Montar e conferir a entrada

Uma linha por janela, com `sensor_id`, `split` e, para cada posição `k` da janela:

| Colunas | Conteúdo | Uso no GPLVM |
|---|---|---|
| `t_0 … t_{k}` | dia de cada célula | mapear o escore de volta para os dias |
| `y_0 … y_{k}` | valor (0 onde não é visível) | dados observados `Y` |
| `mask_0 … mask_{k}` | 1 nas células visíveis | quais células entram na verossimilhança |
| `true_0 … true_{k}` | valor verdadeiro (NaN sem valor) | **só para a métrica** |
| `hidden_0 … hidden_{k}` | 1 nas células escondidas | onde se mede a reconstrução |

Janelas com mais de `MAX_FALTANTE` células sem valor visível são descartadas, assim como a sobra no fim de cada bloco (menos dias que a janela).

In [3]:
JANELA = 7          # dias por janela (7 = semanal; 30 = mensal)
MAX_FALTANTE = 0.5  # descarta janelas com mais de 50% de células sem valor visível

entradas = gplvm_table(serie, window=JANELA, max_missing=MAX_FALTANTE)

gl = gplvm_arrays(entradas)
print("janelas:", len(entradas), "| Y", gl["Y"].shape, " NaN:", np.isnan(gl["Y"]).any())
display(entradas.groupby("split", sort=False).agg(janelas=("sensor_id", "size"),
                                                  estacoes=("sensor_id", "nunique")))

janelas: 164029 | Y (164029, 7)  NaN: False


,janelas,estacoes
split,,
treino,99928,200
validacao,24409,200
calibracao,16250,198
teste,23442,197


## 4. Salvar

Se a entrada já existir com as mesmas escolhas, nada é regravado. Se as escolhas mudarem (outra janela, por exemplo), a célula para com erro: apague a pasta `data/exports/<NOME>/modelo3_gplvm/` para gerar de novo.

In [4]:
save_inputs(entradas, NOME, "modelo3_gplvm", config={
    "estacoes": sorted(entradas["sensor_id"].unique().tolist()),
    "janela": JANELA,
    "max_faltante": MAX_FALTANTE,
})

Entrada já existente, mantida: C:\Users\aliss\OneDrive\Repositorios\usgs_streamflow\data\exports\serie_2010_2025_200est\modelo3_gplvm\janelas.parquet


WindowsPath('C:/Users/aliss/OneDrive/Repositorios/usgs_streamflow/data/exports/serie_2010_2025_200est/modelo3_gplvm/janelas.parquet')

## 5. Como o notebook do modelo lê esta entrada

```python
from src.model_exports import load_inputs, gplvm_arrays
from src.model_inputs import select_split, windows_to_days

entradas, cfg = load_inputs("serie_2010_2025_200est", "modelo3_gplvm")
gl = gplvm_arrays(entradas)

tr, te = select_split(gl, "treino"), select_split(gl, "teste")
Y_train, M_train = tr["Y"], tr["mask"]             # ajuste
Y_test, M_test = te["Y"], te["mask"]               # inferir q(X) e calcular o escore
Y_test_true, H_test = te["Y_true"], te["hidden"]   # só para a métrica

# escore por janela -> por dia: windows_to_days(te, elbo_negativo_por_janela)
```

`gplvm_arrays` devolve o mesmo dicionário que `gplvm_inputs`.

## 6. Forecast

Além de reconstruir e detectar, o modelo precisa **prever à frente**. A tarefa é a mesma para os três modelos (seção 7 do `03_guia_bases_modelos.ipynb`): em cada **origem**, o modelo lê só os dias visíveis até ela e prevê os **horizontes** `h` (dia `origem + h`). Origens e horizontes vêm de `ForecastConfig`; o padrão (`horizons=(1, 7, 30)`, `origin_step=30`) é o da comparação entre modelos, e outros valores podem ser usados como análises adicionais.

**Opção b: dinâmica latente.** As janelas acima são independentes (montadas dentro de cada bloco, com descarte das muito incompletas) e servem para reconstrução e detecção. Para prever, o Modelo 3 precisa da **sequência completa de janelas** de cada estação, em ordem, para ajustar uma dinâmica sobre os códigos latentes (por exemplo, x_{k+1} = f(x_k, sazonalidade)).

`gplvm_sequence_table` monta essa sequência numa grade de calendário comum a todas as estações, sem descartar janelas (as células sem valor ficam com máscara 0). A grade termina no último dia; os primeiros `T % JANELA` dias ficam de fora. Cada janela tem `split` (ou `misto`, se atravessa dois blocos): a dinâmica é ajustada só nas janelas com `split == 'treino'`. A sazonalidade de cada janela (seno e cosseno do dia do ano no início) vem em `sazonal`.

Saída: `data/exports/<NOME>/modelo3_gplvm_dinamica/sequencia.parquet` e `config.json`.

In [5]:
sequencia = gplvm_sequence_table(serie, window=JANELA)
seq = gplvm_sequence_arrays(sequencia)
print("sequência:", seq["Y"].shape, "(estações, janelas, dias) | NaN em Y:", np.isnan(seq["Y"]).any())
print("primeiro dia da grade: t =", seq["t_inicio"][0], "| último: t =", seq["t_fim"][-1])
display(pd.Series(seq["split"]).value_counts().rename("janelas").to_frame())

save_inputs(sequencia, NOME, "modelo3_gplvm_dinamica", config={
    "estacoes": seq["sensors"],
    "janela": JANELA,
    "inicio_grade": int(seq["t_inicio"][0]),
})

sequência: (200, 834, 7) (estações, janelas, dias) | NaN em Y: False
primeiro dia da grade: t = 6 | último: t = 5843


,janelas
treino,500
validacao,125
teste,125
calibracao,82
misto,2


Entrada salva em C:\Users\aliss\OneDrive\Repositorios\usgs_streamflow\data\exports\serie_2010_2025_200est\modelo3_gplvm_dinamica\sequencia.parquet


WindowsPath('C:/Users/aliss/OneDrive/Repositorios/usgs_streamflow/data/exports/serie_2010_2025_200est/modelo3_gplvm_dinamica/sequencia.parquet')

**Forecast na origem.** `gplvm_forecast_inputs` entrega as janelas que começam até a origem, com as células depois dela mascaradas, e, para cada horizonte, a janela futura e a posição do dia previsto dentro dela.

Como o notebook do modelo lê esta entrada:

```python
from src.model_exports import load_inputs, gplvm_sequence_arrays
from src.forecast import ForecastConfig, forecast_origins, gplvm_forecast_inputs

sequencia, cfg = load_inputs("serie_2010_2025_200est", "modelo3_gplvm_dinamica")
seq = gplvm_sequence_arrays(sequencia)          # Y, mask: (S, K, janela); split, sazonal: por janela

treino = seq["split"] == "treino"               # janelas para ajustar o GPLVM e a dinâmica
Y_train, M_train = seq["Y"][:, treino], seq["mask"][:, treino]

FORECAST = ForecastConfig()                     # horizontes ajustáveis
for origem in forecast_origins(serie, FORECAST).query("split == 'teste'")["origem"]:
    f = gplvm_forecast_inputs(seq, origem, FORECAST.horizons)
    # inferir os códigos de f["Y"]/f["mask"], projetar a dinâmica e decodificar f["alvos"]
```

In [6]:
FORECAST = ForecastConfig(horizons=(1, 7, 30), origin_step=30)   # ajustável
origem = int(forecast_origins(serie, FORECAST).query("split == 'teste'")["origem"].iloc[0])
gl_f = gplvm_forecast_inputs(seq, origem, FORECAST.horizons)
print(f"origem t = {origem} | janelas lidas: {gl_f['Y'].shape} | janelas futuras: {len(gl_f['sazonal_futuro'])}")
display(gl_f["alvos"])

origem t = 4966 | janelas lidas: (200, 709, 7) | janelas futuras: 125


,h,t,janela,posicao
0,1,4967,708,5
1,7,4973,709,4
2,30,4996,712,6
